# Landscape plotting with Plotly

Interactive views from `opfunu.plot.Plotting`: a rotatable 3D surface, a
zoomable 2D contour map, custom templates and HTML / image export.

```bash
pip install "opfunu-core[plotly]"
pip install kaleido   # optional: static PNG/PDF/SVG export
```

Rendering is lazy and the sampled grid is cached — exporting several formats
reuses one sampling pass.

In [1]:
from pathlib import Path

from opfunu.cec_based.cec2010 import F12010
from opfunu.plot import HAS_PLOTLY, Plotting

print("plotly available:", HAS_PLOTLY)

f0 = F12010(ndim=10)
figures = Path("figures")
figures.mkdir(exist_ok=True)

plotly available: True


## 1. Interactive 3D surface

Drag to rotate, scroll to zoom, hover for values. Evaluating `surface` as the
last cell expression embeds it inline via the HTML repr; `surface.show()`
works the same.

In [2]:
surface = Plotting(f0.evaluate, backend="plotly", mode="3d", n_points=80)
surface

<Plotting backend='plotly' mode='3d' dims=(1, 2) n_points=80>

## 2. Interactive 2D contour

Plotly additionally offers a `mode="2d"` heatmap with hover readouts and
zoom/pan. The pyvista backend renders 3D meshes only, so use matplotlib or
plotly for two-dimensional views.

In [3]:
contour = Plotting(f0.evaluate, backend="plotly", mode="2d", selected_dims=(2, 3), n_points=200)
contour

<Plotting backend='plotly' mode='2d' dims=(2, 3) n_points=200>

## 3. Colormaps and templates

`cmap` names are mapped onto built-in Plotly colorscales where they match, and
`template` takes a registered template name, a template dict or a
`go.layout.Template`.

In [4]:
Plotting(
    f0.evaluate,
    backend="plotly",
    selected_dims=(1, 6),
    cmap="cividis",
    template="plotly_white",
    title="F12010 — dims (1, 6), plotly_white",
).show()

In [5]:
import plotly.io as pio

pio.templates["paper_slice"] = {
    "layout": {"font": {"family": "Times New Roman", "size": 12}},
}

themed = Plotting(f0.evaluate, backend="plotly", template="paper_slice", n_points=80)
themed

<Plotting backend='plotly' mode='3d' dims=(1, 2) n_points=80>

## 4. Export

`.html` always works (the plotly.js bundle comes from the CDN by default —
pass `include_plotlyjs=True` for a fully offline file). Static images go
through `kaleido`; without it, `save()` raises a guiding `ImportError`.

In [6]:
surface.save(figures / "landscape3d.html")
contour.save(figures / "landscape2d.html", include_plotlyjs=True)  # offline-ready file

PosixPath('figures/landscape2d.html')

In [7]:
try:
    surface.save(figures / "landscape3d.png", scale=2)
except ImportError:
    print("static image export needs kaleido: pip install kaleido")

static image export needs kaleido: pip install kaleido


## 5. Native figure access

`plot.figure` exposes the `plotly.graph_objects.Figure`; layout tweaks apply
to later renders and exports.

In [8]:
surface.figure.update_layout(width=800, height=600)
surface.save(figures / "landscape3d_wide.html")

PosixPath('figures/landscape3d_wide.html')

## 6. Cleanup

Plotly figures need no explicit teardown, but `close()` keeps the lifecycle
identical across backends (the sampled grid stays cached).

In [9]:
surface.close()
contour.close()
themed.close()